In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [2]:
train_data = pd.read_csv('../data/ProcessedDataForTestModel/historical_train_data.csv')
prediction_data = pd.read_csv('../data/ProcessedDataForTestModel/prediction_data.csv')

In [3]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6224 entries, 0 to 6223
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  6224 non-null   int64  
 1   date        6224 non-null   str    
 2   home_team   6224 non-null   str    
 3   away_team   6224 non-null   str    
 4   home_score  6224 non-null   float64
 5   away_score  6224 non-null   float64
 6   tournament  6224 non-null   str    
 7   city        6224 non-null   str    
 8   country     6224 non-null   str    
 9   neutral     6224 non-null   bool   
 10  year        6224 non-null   int64  
dtypes: bool(1), float64(2), int64(2), str(6)
memory usage: 492.5 KB


In [4]:
prediction_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  72 non-null     int64  
 1   date        72 non-null     str    
 2   home_team   72 non-null     str    
 3   away_team   72 non-null     str    
 4   home_score  0 non-null      float64
 5   away_score  0 non-null      float64
 6   tournament  72 non-null     str    
 7   city        72 non-null     str    
 8   country     72 non-null     str    
 9   neutral     72 non-null     bool   
 10  year        72 non-null     int64  
dtypes: bool(1), float64(2), int64(2), str(6)
memory usage: 5.8 KB


In [5]:
prediction_data.drop(columns=['Unnamed: 0'], inplace=True)
prediction_data

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,2026
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,2026
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,2026
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,2026
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,2026
...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,2026
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,2026
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,2026
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,2026


In [6]:
train_data.drop(columns=['Unnamed: 0'], inplace= True)

In [7]:
train_data

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year
0,2000-03-04,Honduras,Nicaragua,3.0,0.0,FIFA World Cup qualification,San Pedro Sula,Honduras,False,2000
1,2000-03-04,Trinidad and Tobago,Curaçao,5.0,0.0,FIFA World Cup qualification,Macoya,Trinidad and Tobago,False,2000
2,2000-03-05,Anguilla,Bahamas,1.0,3.0,FIFA World Cup qualification,The Valley,Anguilla,False,2000
3,2000-03-05,Barbados,Grenada,2.0,2.0,FIFA World Cup qualification,Bridgetown,Barbados,False,2000
4,2000-03-05,British Virgin Islands,Bermuda,1.0,5.0,FIFA World Cup qualification,Road Town,British Virgin Islands,False,2000
...,...,...,...,...,...,...,...,...,...,...
6219,2026-03-31,Iraq,Bolivia,2.0,1.0,FIFA World Cup qualification,Guadalupe,Mexico,True,2026
6220,2026-03-31,Bosnia and Herzegovina,Italy,1.0,1.0,FIFA World Cup qualification,Zenica,Bosnia and Herzegovina,False,2026
6221,2026-03-31,Sweden,Poland,3.0,2.0,FIFA World Cup qualification,Solna,Sweden,False,2026
6222,2026-03-31,Kosovo,Turkey,0.0,1.0,FIFA World Cup qualification,Pristina,Kosovo,False,2026


In [8]:
## Prepairing the data for training and testing
from torch.utils.data import DataLoader, TensorDataset

In [9]:
historical_match_data = train_data.copy()
historical_match_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6224 entries, 0 to 6223
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        6224 non-null   str    
 1   home_team   6224 non-null   str    
 2   away_team   6224 non-null   str    
 3   home_score  6224 non-null   float64
 4   away_score  6224 non-null   float64
 5   tournament  6224 non-null   str    
 6   city        6224 non-null   str    
 7   country     6224 non-null   str    
 8   neutral     6224 non-null   bool   
 9   year        6224 non-null   int64  
dtypes: bool(1), float64(2), int64(1), str(6)
memory usage: 443.8 KB


In [10]:
## Sorting form the oldest to newest matches in the dataset and adding the type of tournament columns (is_world_cup, is_qualifier)
prediction_data['date'] = pd.to_datetime(prediction_data['date'])
historical_match_data['date'] = pd.to_datetime(historical_match_data['date'])
historical_match_data = historical_match_data.sort_values('date').reset_index(drop=True)

historical_match_data['neutral'] = historical_match_data['neutral'].astype(int)

historical_match_data['is_world_cup'] = (historical_match_data['tournament']=='FIFA World Cup').astype(int) ## Meaning: 1 is World Cup match, 0 is not
historical_match_data['is_qualifier'] = (historical_match_data['tournament'] == 'FIFA World Cup qualification').astype(int) ## Meaning: same logic as the world cup column

historical_match_data.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,is_world_cup,is_qualifier
0,2000-03-04,Honduras,Nicaragua,3.0,0.0,FIFA World Cup qualification,San Pedro Sula,Honduras,0,2000,0,1
1,2000-03-04,Trinidad and Tobago,Curaçao,5.0,0.0,FIFA World Cup qualification,Macoya,Trinidad and Tobago,0,2000,0,1
2,2000-03-05,Saint Lucia,Suriname,1.0,0.0,FIFA World Cup qualification,Castries,Saint Lucia,0,2000,0,1
3,2000-03-05,El Salvador,Belize,5.0,0.0,FIFA World Cup qualification,San Salvador,El Salvador,0,2000,0,1
4,2000-03-05,Dominican Republic,Montserrat,3.0,0.0,FIFA World Cup qualification,San Cristóbal,Dominican Republic,0,2000,0,1


In [11]:
## Splitting the data chronologically (train, validation, test)
n = len(historical_match_data)

train_end = int(0.70 * n) ## Index of last row used for training
val_end = int (0.85 * n) ## index of last row used for validation

train_cutoff_date = historical_match_data.loc[train_end - 1, 'date'] ## The last match used for training => that date
val_cutoff_date = historical_match_data.loc[val_end - 1, 'date'] ## The last match used for validation

train_df = historical_match_data[historical_match_data['date'] <= train_cutoff_date].copy()

val_df = historical_match_data[(historical_match_data['date'] > train_cutoff_date) & (historical_match_data['date'] <= val_cutoff_date)].copy()

test_df = historical_match_data[historical_match_data['date'] > val_cutoff_date].copy()

print("Train:", len(train_df), train_df["date"].min(), "to", train_df["date"].max())
print("Val:  ", len(val_df), val_df["date"].min(), "to", val_df["date"].max())
print("Test: ", len(test_df), test_df["date"].min(), "to", test_df["date"].max())

## The idea was to split the dataset into train data that are older matches, a validation set from 2018 - 2022 and a test set from 2022 - 2026 so we dont train on newer matches and test on older

Train: 4358 2000-03-04 00:00:00 to 2018-06-23 00:00:00
Val:   933 2018-06-24 00:00:00 to 2022-11-28 00:00:00
Test:  933 2022-11-29 00:00:00 to 2026-03-31 00:00:00


In [12]:
## Encoding the teams => transforming team names into numbers

all_teams = sorted(set(train_df['home_team']).union(set(train_df['away_team'])))

team_to_id = {team: idx for idx, team in enumerate(all_teams)}
id_to_team = {idx: team for team, idx in team_to_id.items()}

num_teams = len(team_to_id)

print("Number of teams: ", num_teams)

for df in [train_df, val_df, test_df]:
    df['home_team_id'] = df['home_team'].map(team_to_id)
    df['away_team_id'] = df['away_team'].map(team_to_id)
    
print(train_df[["home_team", "home_team_id", "away_team", "away_team_id"]].head())

Number of teams:  211
             home_team  home_team_id   away_team  away_team_id
0             Honduras            84   Nicaragua           134
1  Trinidad and Tobago           192     Curaçao            49
2          Saint Lucia           158    Suriname           179
3          El Salvador            59      Belize            20
4   Dominican Republic            56  Montserrat           125


In [13]:
## Checking if any team is missing in any data split

print("Missing team IDs in train:")
print(train_df[["home_team_id", "away_team_id"]].isna().sum())

print("Missing team IDs in val:")
print(val_df[["home_team_id", "away_team_id"]].isna().sum())

print("Missing team IDs in test:")
print(test_df[["home_team_id", "away_team_id"]].isna().sum())

Missing team IDs in train:
home_team_id    0
away_team_id    0
dtype: int64
Missing team IDs in val:
home_team_id    0
away_team_id    0
dtype: int64
Missing team IDs in test:
home_team_id    0
away_team_id    0
dtype: int64


In [14]:
## Normalize using the training data
year_mean = train_df['year'].mean()
year_std = train_df['year'].std()

for df in [train_df, val_df, test_df]:
    df['year_norm'] = (df['year'] - year_mean) / year_std

In [15]:
## Defining the numeric features
numeric_feature_columns = ['neutral', 'is_world_cup', 'is_qualifier']
num_numeric_features = len(numeric_feature_columns)

print("Number of numeric features:", num_numeric_features)

Number of numeric features: 3


In [16]:
## Adding year_norm to the num features
numeric_feature_columns_with_year = ['year_norm', 'neutral', 'is_world_cup', 'is_qualifier']
num_numeric_features_with_year = len(numeric_feature_columns_with_year)

print("Number of numeric features:", num_numeric_features_with_year)

Number of numeric features: 4


In [17]:
## Creating PyTorch datasets
def make_dataset(df):
    X_home = torch.tensor(
        df['home_team_id'].values,
        dtype=torch.long
    )

    X_away = torch.tensor(
        df['away_team_id'].values,
        dtype=torch.long
    )

    X_num = torch.tensor(
        df[numeric_feature_columns].values.astype(np.float32),
        dtype=torch.float32
    )

    y = torch.tensor(
        df[['home_score', 'away_score']].values.astype(np.float32),
        dtype = torch.float32
    )

    return TensorDataset(X_home, X_away, X_num, y)

train_dataset = make_dataset(train_df)
val_dataset = make_dataset(val_df)
test_dataset = make_dataset(test_df)


In [18]:
## Creating PyTorch dataset for num features with year_norm
def make_dataset(df):
    X_home = torch.tensor(
        df['home_team_id'].values,
        dtype=torch.long
    )

    X_away = torch.tensor(
        df['away_team_id'].values,
        dtype=torch.long
    )

    X_num = torch.tensor(
        df[numeric_feature_columns_with_year].values.astype(np.float32),
        dtype=torch.float32
    )

    y = torch.tensor(
        df[['home_score', 'away_score']].values.astype(np.float32),
        dtype = torch.float32
    )

    return TensorDataset(X_home, X_away, X_num, y)

train_dataset_with_year = make_dataset(train_df)
val_dataset_with_year = make_dataset(val_df)
test_dataset_with_year = make_dataset(test_df)

In [19]:
## Creating the DataLoaders

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train matches:", len(train_dataset))
print("Val matches:", len(val_dataset))
print("Test matches:", len(test_dataset))

Train matches: 4358
Val matches: 933
Test matches: 933


In [40]:
## Creating DataLoaders with year_norm

batch_size = 32

train_loader_with_year = DataLoader(
    train_dataset_with_year,
    batch_size=batch_size,
    shuffle=True
)

val_loader_with_year = DataLoader(
    val_dataset_with_year,
    batch_size=batch_size,
    shuffle=False
)

test_loader_with_year = DataLoader(
    test_dataset_with_year,
    batch_size=batch_size,
    shuffle=False
)

print("Train matches:", len(train_dataset_with_year))
print("Val matches:", len(val_dataset_with_year))
print("Test matches:", len(test_dataset_with_year))

Train matches: 4358
Val matches: 933
Test matches: 933


In [21]:
for home_ids, away_ids, numeric, y in train_loader:
    print("home_ids shape:", home_ids.shape)
    print("away_ids shape:", away_ids.shape)
    print("numeric shape:", numeric.shape)
    print("target shape:", y.shape)
    break

home_ids shape: torch.Size([32])
away_ids shape: torch.Size([32])
numeric shape: torch.Size([32, 3])
target shape: torch.Size([32, 2])


In [22]:
## Shapes for the second model
for home_ids, away_ids, numeric, y in train_loader_with_year:
    print("home_ids shape:", home_ids.shape)
    print("away_ids shape:", away_ids.shape)
    print("numeric shape:", numeric.shape)
    print("target shape:", y.shape)
    break

home_ids shape: torch.Size([32])
away_ids shape: torch.Size([32])
numeric shape: torch.Size([32, 4])
target shape: torch.Size([32, 2])


In [23]:
##Defining the model
import torch.nn.functional as F
class ExpectedGoalsModel(nn.Module):
    def __init__(self, num_teams, embedding_dim, num_numeric_features):
        super().__init__()

        self.team_embedding = nn.Embedding(
            num_embeddings=num_teams,
            embedding_dim=embedding_dim
        )

        input_dim = 2 * embedding_dim + num_numeric_features

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, home_team_ids, away_team_ids, numeric_features):
        home_emb = self.team_embedding(home_team_ids)
        away_emb = self.team_embedding(away_team_ids)

        x = torch.cat(
            [home_emb, away_emb, numeric_features], dim=1
        )

        raw_output = self.network(x)

        expected_goals = F.softplus(raw_output)

        return(expected_goals)

In [24]:
## Creating the model:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device: ", device)

model = ExpectedGoalsModel(
    num_teams=num_teams,
    embedding_dim = 4,
    num_numeric_features = num_numeric_features
).to(device)


Using device:  cuda


In [41]:
## Creating the second model for comparisson
device_v2 = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device: ", device_v2)

model_v2 = ExpectedGoalsModel(
    num_teams=num_teams,
    embedding_dim = 4,
    num_numeric_features = num_numeric_features_with_year
).to(device)

Using device:  cuda


In [26]:
## Setting up the loss function and optimizer
import torch.optim as optim

loss_fn = nn.PoissonNLLLoss(
    log_input=False,
    full=True
)

optimizer = optim.Adam(
    model.parameters(),
    lr = 1e-3,
    weight_decay=1e-4
)
num_epochs = 200

In [42]:
## Optimizer and loss for the model v2
import torch.optim as optim

loss_fn = nn.PoissonNLLLoss(
    log_input=False,
    full=True
)

optimizer_v2 = optim.Adam(
    model_v2.parameters(),
    lr = 1e-3,
    weight_decay=1e-4
)
num_epochs_v2 = 200

In [28]:
## Training the mdoel
import copy

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_model_state = None

for epoch in range(num_epochs):
    model.train()

    total_train_loss = 0.0
    num_train_batches = 0

    for home_ids, away_ids, numeric, y in train_loader:
        home_ids = home_ids.to(device)
        away_ids = away_ids.to(device)
        numeric = numeric.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        predictions = model(home_ids, away_ids, numeric)

        loss = loss_fn(predictions, y)

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()
        num_train_batches += 1

    avg_train_loss = total_train_loss / num_train_batches

    model.eval()

    total_val_loss = 0.0
    num_val_batches = 0

    with torch.no_grad():
        for home_ids, away_ids, numeric, y in val_loader:
            home_ids = home_ids.to(device)
            away_ids = away_ids.to(device)
            numeric = numeric.to(device)
            y = y.to(device)

            predictions = model(home_ids, away_ids, numeric)

            loss = loss_fn(predictions, y)

            total_val_loss += loss.item()
            num_val_batches += 1

    avg_val_loss = total_val_loss / num_val_batches

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch

    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

Epoch 000 | Train Loss: 1.7462 | Val Loss: 1.6848
Epoch 010 | Train Loss: 1.5379 | Val Loss: 1.5344
Epoch 020 | Train Loss: 1.4159 | Val Loss: 1.4472
Epoch 030 | Train Loss: 1.3774 | Val Loss: 1.4356
Epoch 040 | Train Loss: 1.3592 | Val Loss: 1.4332
Epoch 050 | Train Loss: 1.3442 | Val Loss: 1.4347
Epoch 060 | Train Loss: 1.3370 | Val Loss: 1.4353
Epoch 070 | Train Loss: 1.3307 | Val Loss: 1.4426
Epoch 080 | Train Loss: 1.3241 | Val Loss: 1.4460
Epoch 090 | Train Loss: 1.3185 | Val Loss: 1.4508
Epoch 100 | Train Loss: 1.3133 | Val Loss: 1.4529
Epoch 110 | Train Loss: 1.3081 | Val Loss: 1.4537
Epoch 120 | Train Loss: 1.3051 | Val Loss: 1.4564
Epoch 130 | Train Loss: 1.3027 | Val Loss: 1.4572
Epoch 140 | Train Loss: 1.3010 | Val Loss: 1.4614
Epoch 150 | Train Loss: 1.2966 | Val Loss: 1.4629
Epoch 160 | Train Loss: 1.2928 | Val Loss: 1.4663
Epoch 170 | Train Loss: 1.2902 | Val Loss: 1.4704
Epoch 180 | Train Loss: 1.2878 | Val Loss: 1.4766
Epoch 190 | Train Loss: 1.2868 | Val Loss: 1.4810


In [29]:
## After training and memorizing the best epoch number we are restoring the best model
model.load_state_dict(best_model_state)

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

Best epoch: 39
Best validation loss: 1.4308193564414977


In [43]:
## Training the second model
import copy

train_losses_v2 = []
val_losses_v2 = []

best_val_loss_v2 = float("inf")
best_model_state_v2 = None

for epoch in range(num_epochs_v2):
    model_v2.train()

    total_train_loss = 0.0
    num_train_batches = 0

    for home_ids, away_ids, numeric, y in train_loader_with_year:
        home_ids = home_ids.to(device)
        away_ids = away_ids.to(device)
        numeric = numeric.to(device)
        y = y.to(device)

        optimizer_v2.zero_grad()

        predictions = model_v2(home_ids, away_ids, numeric)

        loss = loss_fn(predictions, y)

        loss.backward()

        optimizer_v2.step()

        total_train_loss += loss.item()
        num_train_batches += 1

    avg_train_loss = total_train_loss / num_train_batches

    model_v2.eval()

    total_val_loss = 0.0
    num_val_batches = 0

    with torch.no_grad():
        for home_ids, away_ids, numeric, y in val_loader_with_year:
            home_ids = home_ids.to(device)
            away_ids = away_ids.to(device)
            numeric = numeric.to(device)
            y = y.to(device)

            predictions = model_v2(home_ids, away_ids, numeric)

            loss = loss_fn(predictions, y)

            total_val_loss += loss.item()
            num_val_batches += 1

    avg_val_loss = total_val_loss / num_val_batches

    train_losses_v2.append(avg_train_loss)
    val_losses_v2.append(avg_val_loss)

    if avg_val_loss < best_val_loss_v2:
        best_val_loss_v2 = avg_val_loss
        best_model_state_v2 = copy.deepcopy(model_v2.state_dict())
        best_epoch = epoch

    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss_v2)

Epoch 000 | Train Loss: 1.7673 | Val Loss: 1.6972
Epoch 010 | Train Loss: 1.5150 | Val Loss: 1.5167
Epoch 020 | Train Loss: 1.4165 | Val Loss: 1.4488
Epoch 030 | Train Loss: 1.3789 | Val Loss: 1.4289
Epoch 040 | Train Loss: 1.3648 | Val Loss: 1.4254
Epoch 050 | Train Loss: 1.3475 | Val Loss: 1.4260
Epoch 060 | Train Loss: 1.3384 | Val Loss: 1.4250
Epoch 070 | Train Loss: 1.3329 | Val Loss: 1.4330
Epoch 080 | Train Loss: 1.3210 | Val Loss: 1.4383
Epoch 090 | Train Loss: 1.3149 | Val Loss: 1.4458
Epoch 100 | Train Loss: 1.3076 | Val Loss: 1.4550
Epoch 110 | Train Loss: 1.3034 | Val Loss: 1.4581
Epoch 120 | Train Loss: 1.2983 | Val Loss: 1.4652
Epoch 130 | Train Loss: 1.2919 | Val Loss: 1.4777
Epoch 140 | Train Loss: 1.2883 | Val Loss: 1.4763
Epoch 150 | Train Loss: 1.2846 | Val Loss: 1.4770
Epoch 160 | Train Loss: 1.2823 | Val Loss: 1.4825
Epoch 170 | Train Loss: 1.2781 | Val Loss: 1.5083
Epoch 180 | Train Loss: 1.2748 | Val Loss: 1.4926
Epoch 190 | Train Loss: 1.2721 | Val Loss: 1.5249


In [44]:
## After training and memorizing the best epoch number we are restoring the best model v2
model_v2.load_state_dict(best_model_state_v2)

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

Best epoch: 35
Best validation loss: 1.4308193564414977


In [45]:
## Evaluating the model on the chronological test set:
model.eval()

total_test_loss = 0.0
num_test_batches = 0

with torch.no_grad():
    for home_ids, away_ids, numeric, y in test_loader:
        home_ids = home_ids.to(device)
        away_ids = away_ids.to(device)
        numeric = numeric.to(device)
        y = y.to(device)

        predictions = model(home_ids, away_ids, numeric)

        loss = loss_fn(predictions, y)

        total_test_loss += loss.item()
        num_test_batches += 1

avg_test_loss = total_test_loss / num_test_batches

print("Test loss:", avg_test_loss)

Test loss: 1.4677685181299844


In [51]:
## Evaluating the model v2 on the chronological test set:
model_v2.eval()

total_test_loss = 0.0
num_test_batches = 0

with torch.no_grad():
    for home_ids, away_ids, numeric, y in test_loader_with_year:
        home_ids = home_ids.to(device)
        away_ids = away_ids.to(device)
        numeric = numeric.to(device)
        y = y.to(device)

        predictions = model_v2(home_ids, away_ids, numeric)

        loss = loss_fn(predictions, y)

        total_test_loss += loss.item()
        num_test_batches += 1

avg_test_loss = total_test_loss / num_test_batches

print("Test loss:", avg_test_loss)

Test loss: 1.4822711229324341


In [52]:
## Inspecting actual predictions from the test set
model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for home_ids, away_ids, numeric, y in test_loader:
        home_ids = home_ids.to(device)
        away_ids = away_ids.to(device)
        numeric = numeric.to(device)

        predictions = model(home_ids, away_ids, numeric)

        all_predictions.append(predictions.cpu())
        all_targets.append(y)

all_predictions = torch.cat(all_predictions, dim=0).numpy()
all_targets = torch.cat(all_targets, dim=0).numpy()

results_df = test_df.copy()
results_df["pred_home_goals"] = all_predictions[:, 0]
results_df["pred_away_goals"] = all_predictions[:, 1]
results_df["actual_home_goals"] = all_targets[:, 0]
results_df["actual_away_goals"] = all_targets[:, 1]

results_df[
    [
        "date",
        "home_team",
        "away_team",
        "actual_home_goals",
        "actual_away_goals",
        "pred_home_goals",
        "pred_away_goals",
        "tournament",
        "neutral",
    ]
].head(20)

,date,home_team,away_team,actual_home_goals,actual_away_goals,pred_home_goals,pred_away_goals,tournament,neutral
5291,2022-11-29,Qatar,Netherlands,0.0,2.0,1.284586,1.160818,FIFA World Cup,0
5292,2022-11-29,Ecuador,Senegal,1.0,2.0,1.005113,1.748485,FIFA World Cup,1
5293,2022-11-29,Wales,England,0.0,3.0,0.839439,1.481188,FIFA World Cup,1
5294,2022-11-29,Iran,United States,0.0,1.0,0.826306,1.465764,FIFA World Cup,1
5295,2022-11-30,Poland,Argentina,0.0,2.0,1.156357,1.292732,FIFA World Cup,1
5296,2022-11-30,Saudi Arabia,Mexico,1.0,2.0,1.138920,1.268863,FIFA World Cup,1
5297,2022-11-30,Australia,Denmark,1.0,0.0,1.694644,1.097086,FIFA World Cup,1
5298,2022-11-30,Tunisia,France,1.0,0.0,0.916312,1.593492,FIFA World Cup,1
5299,2022-12-01,Canada,Morocco,1.0,2.0,1.782388,0.597126,FIFA World Cup,1
5300,2022-12-01,Croatia,Belgium,0.0,0.0,1.160670,1.124743,FIFA World Cup,1


In [53]:
#Inspecting actual predictions from the test set for model v2
model_v2.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for home_ids, away_ids, numeric, y in test_loader_with_year:
        home_ids = home_ids.to(device)
        away_ids = away_ids.to(device)
        numeric = numeric.to(device)

        predictions = model_v2(home_ids, away_ids, numeric)

        all_predictions.append(predictions.cpu())
        all_targets.append(y)

all_predictions = torch.cat(all_predictions, dim=0).numpy()
all_targets = torch.cat(all_targets, dim=0).numpy()

results_df = test_df.copy()
results_df["pred_home_goals"] = all_predictions[:, 0]
results_df["pred_away_goals"] = all_predictions[:, 1]
results_df["actual_home_goals"] = all_targets[:, 0]
results_df["actual_away_goals"] = all_targets[:, 1]

results_df[
    [
        "date",
        "home_team",
        "away_team",
        "actual_home_goals",
        "actual_away_goals",
        "pred_home_goals",
        "pred_away_goals",
        "tournament",
        "neutral",
    ]
].head(20)

,date,home_team,away_team,actual_home_goals,actual_away_goals,pred_home_goals,pred_away_goals,tournament,neutral
5291,2022-11-29,Qatar,Netherlands,0.0,2.0,0.772209,1.719470,FIFA World Cup,0
5292,2022-11-29,Ecuador,Senegal,1.0,2.0,1.278698,1.461258,FIFA World Cup,1
5293,2022-11-29,Wales,England,0.0,3.0,0.665913,2.554808,FIFA World Cup,1
5294,2022-11-29,Iran,United States,0.0,1.0,1.800738,0.780041,FIFA World Cup,1
5295,2022-11-30,Poland,Argentina,0.0,2.0,0.873328,1.862087,FIFA World Cup,1
5296,2022-11-30,Saudi Arabia,Mexico,1.0,2.0,1.150477,1.586714,FIFA World Cup,1
5297,2022-11-30,Australia,Denmark,1.0,0.0,2.317267,0.648111,FIFA World Cup,1
5298,2022-11-30,Tunisia,France,1.0,0.0,1.703633,0.894543,FIFA World Cup,1
5299,2022-12-01,Canada,Morocco,1.0,2.0,1.081687,1.546531,FIFA World Cup,1
5300,2022-12-01,Croatia,Belgium,0.0,0.0,1.259084,1.292223,FIFA World Cup,1


In [ ]:
## In conclusion, the model_v2 with year_norm column is practically tied with the simpler model version without year_norm. Depending on the randomness sometimes the model wins sometimes it losses so
## we are going to keep both of these versions in the notebook but the simpler (without year_norm) model will be the Demo_v2 baseline.

In [ ]:
## For the next model we should use a shorter time period and we are going to add more features to the dataset